# DistilBERT (Colab) — Experiment B

Same **training-size benchmark** as `04_distilbert_colab.ipynb` (Jigsaw multi-label, `pos_weight`, per-label threshold tuning, CSV artifacts on Drive), with **Experiment B** training changes:

- **Up to 5 epochs** per train size (`MAX_EPOCHS = 5`).
- **Linear warmup + linear decay** LR schedule via Hugging Face `get_linear_schedule_with_warmup` (warmup as a fraction of total optimizer steps).
- **Early stopping** on validation loss (patience in epochs, optional `min_delta`; restores best checkpoint before threshold tuning / metrics).

Artifacts go to `notebooks/artifacts_colab_exp_b/` so runs do not overwrite `artifacts_colab/`.

### Using an L4 on Colab (speed)

This notebook enables **bf16 mixed precision** when CUDA reports bf16 support (L4 does). You can also:

- **Raise `BATCH_SIZE`** until GPU memory is high but stable (try 32 → 48 → 64 at `MAX_LENGTH=128`).
- Set **`USE_TORCH_COMPILE = True`** (PyTorch 2.x) after `model.to(device)`; if it errors, set back to `False`.
- Use **`num_workers`** in `DataLoader` (try 2–4); if Colab workers misbehave, set `NUM_WORKERS = 0`.
- Keep **`TOKENIZERS_PARALLELISM=false`** when using dataloader workers to avoid oversubscription.

Optional **`GRADIENT_ACCUMULATION_STEPS` > 1`** simulates a larger effective batch without raising memory (slower per step but can improve stability).


In [ ]:
# Colab setup: install dependencies
!pip -q install torch transformers pandas numpy matplotlib scikit-learn


In [ ]:
# Mount Google Drive and set paths
from pathlib import Path
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ===== CHANGE THIS to your repo folder in Drive =====
PROJECT_ROOT = Path('/content/drive/MyDrive/cmpe258/jigsaw-toxic-comment-classification-challenge')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'PROJECT_ROOT not found: {PROJECT_ROOT}. Upload/clone your repo to this path first.'
    )

NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
ARTIFACT_DIR = NOTEBOOKS_DIR / 'artifacts_colab_exp_b'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(NOTEBOOKS_DIR))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('ARTIFACT_DIR:', ARTIFACT_DIR)


In [ ]:
import contextlib
import math
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

from preprocessing.text_preprocessing import LABEL_COLUMNS, preprocess_for_distilbert
from metrics_helpers import multilabel_evaluation_report, torch_parameter_count

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def binary_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    denom = 2 * tp + fp + fn
    return float((2 * tp) / denom) if denom > 0 else 0.0


def tune_per_label_thresholds(y_true: np.ndarray, y_prob: np.ndarray, labels, threshold_grid: np.ndarray):
    best_thresholds = {}
    rows = []
    for j, label in enumerate(labels):
        y_true_j = y_true[:, j]
        y_prob_j = y_prob[:, j]
        best_t = 0.5
        best_f1 = -1.0
        for t in threshold_grid:
            y_pred_j = (y_prob_j >= t).astype(np.int64)
            f1_j = binary_f1(y_true_j, y_pred_j)
            if f1_j > best_f1:
                best_f1 = f1_j
                best_t = float(t)
        best_thresholds[label] = best_t
        rows.append({'label': label, 'best_threshold': round(best_t, 3), 'best_f1_on_val': best_f1})
    return best_thresholds, pd.DataFrame(rows)


def apply_thresholds(y_prob: np.ndarray, labels, thresholds: dict[str, float]) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=np.int64)
    for j, label in enumerate(labels):
        out[:, j] = (y_prob[:, j] >= thresholds[label]).astype(np.int64)
    return out


def compute_pos_weight(y_train: np.ndarray, eps: float = 1e-6, max_weight: float = 50.0) -> torch.Tensor:
    y = np.asarray(y_train, dtype=np.float32)
    positives = y.sum(axis=0)
    total = float(y.shape[0])
    negatives = total - positives
    weights = negatives / np.maximum(positives, eps)
    weights = np.clip(weights, 1.0, max_weight)
    return torch.tensor(weights, dtype=torch.float32)


def enc_dict(enc):
    keys = [k for k in ('input_ids', 'attention_mask') if k in enc]
    return {k: enc[k] for k in keys}


class EncDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = self.labels[i]
        return item


def collate(batch):
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0] if k != 'labels'}
    out['labels'] = torch.stack([b['labels'] for b in batch], dim=0)
    return out


In [ ]:
# Experiment B + benchmark configuration
DEVICE = pick_device()
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.manual_seed(42)
np.random.seed(42)

MAX_LENGTH = 128
BATCH_SIZE = 32          # L4: try 48 or 64 if memory allows
MAX_EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01    # standard for AdamW + transformers fine-tuning
WARMUP_RATIO = 0.1     # fraction of total training steps for linear warmup

EARLY_STOP_PATIENCE = 2       # epochs without val-loss improvement
EARLY_STOP_MIN_DELTA = 0.0    # require val_loss < best - min_delta
MAX_GRAD_NORM = 1.0           # set None to disable gradient clipping

GRADIENT_ACCUMULATION_STEPS = 1   # >1 for larger effective batch on same memory

USE_BF16 = True           # L4: keep True when CUDA bf16 is supported
USE_TORCH_COMPILE = False # set True on PyTorch 2.x + L4 if stable
NUM_WORKERS = 2         # Colab: 0 if worker issues

SIZE_STEP = 10000
THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01)
MAX_TRAIN_CAP = None
MAX_VAL_SAMPLES = None
SKIP_COMPLETED = True

_bf16_ok = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
USE_AMP = bool(USE_BF16 and _bf16_ok)
if USE_BF16 and DEVICE.type == 'cuda' and not _bf16_ok:
    print('USE_BF16 requested but bf16 not supported on this GPU; training in full precision.')

probe = preprocess_for_distilbert(
    validation_fraction=0.1,
    random_state=42,
    max_length=MAX_LENGTH,
    return_tensors='pt',
    max_train_samples=MAX_TRAIN_CAP,
    max_val_samples=MAX_VAL_SAMPLES,
)
max_train_available = len(probe.y_train)

if max_train_available < SIZE_STEP:
    train_sizes = [max_train_available]
else:
    train_sizes = list(range(SIZE_STEP, max_train_available + 1, SIZE_STEP))
    if train_sizes[-1] != max_train_available:
        train_sizes.append(max_train_available)

print('Device:', DEVICE)
print('AMP (bf16):', USE_AMP)
print('max_train_available:', max_train_available)
print('train_sizes:', train_sizes[:5], '...', train_sizes[-1], f'(n={len(train_sizes)})')


In [ ]:
# Run benchmark sweep (Experiment B: scheduler + early stopping)
summary_rows = []
per_label_frames = []
threshold_frames = []

pin = DEVICE.type == 'cuda'

for idx, train_size in enumerate(train_sizes, start=1):
    row_file = ARTIFACT_DIR / f'summary_{train_size}.csv'
    per_label_file = ARTIFACT_DIR / f'per_label_{train_size}.csv'
    thresholds_file = ARTIFACT_DIR / f'thresholds_{train_size}.csv'

    if SKIP_COMPLETED and row_file.exists() and per_label_file.exists() and thresholds_file.exists():
        print(f'[{idx}/{len(train_sizes)}] train_size={train_size}: skipping (already exists)')
        summary_rows.append(pd.read_csv(row_file).iloc[0].to_dict())
        per_label_frames.append(pd.read_csv(per_label_file))
        threshold_frames.append(pd.read_csv(thresholds_file))
        continue

    print(f'[{idx}/{len(train_sizes)}] train_size={train_size}: running')
    run_data = preprocess_for_distilbert(
        validation_fraction=0.1,
        random_state=42,
        max_length=MAX_LENGTH,
        return_tensors='pt',
        max_train_samples=train_size,
        max_val_samples=MAX_VAL_SAMPLES,
    )

    train_enc = enc_dict(run_data.train_encodings)
    val_enc = enc_dict(run_data.val_encodings)
    y_train = np.asarray(run_data.y_train, dtype=np.float32)
    y_val = np.asarray(run_data.y_val, dtype=np.int64)

    train_loader = DataLoader(
        EncDataset(train_enc, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
        persistent_workers=NUM_WORKERS > 0,
    )
    val_loader = DataLoader(
        EncDataset(val_enc, y_val),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
        persistent_workers=NUM_WORKERS > 0,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        'distilbert-base-uncased',
        num_labels=len(LABEL_COLUMNS),
        problem_type='multi_label_classification',
    ).to(DEVICE)
    if USE_TORCH_COMPILE and hasattr(torch, 'compile'):
        try:
            model = torch.compile(model)  # type: ignore[assignment]
            print('  torch.compile enabled')
        except Exception as e:
            print('  torch.compile skipped:', e)

    num_params = torch_parameter_count(model)
    pos_weight = compute_pos_weight(y_train).to(DEVICE)
    pos_weight_dict = {label: float(pos_weight[j].item()) for j, label in enumerate(LABEL_COLUMNS)}

    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            'weight_decay': WEIGHT_DECAY,
        },
        {
            'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0,
        },
    ]
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LR)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
    num_training_steps = steps_per_epoch * MAX_EPOCHS
    warmup_steps = int(WARMUP_RATIO * num_training_steps)
    warmup_steps = max(0, min(warmup_steps, num_training_steps - 1)) if num_training_steps > 0 else 0
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=num_training_steps
    )

    train_time_s = 0.0
    inference_time_s = 0.0
    train_loss_last = float('nan')
    val_loss_last = float('nan')

    best_val_loss = float('inf')
    best_state_cpu = None
    best_epoch = 0
    patience_left = EARLY_STOP_PATIENCE
    epochs_ran = 0
    early_stopped = False

    if USE_AMP:
        autocast_ctx = torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=True)
    else:
        autocast_ctx = contextlib.nullcontext()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_train_loss = 0.0
        epoch_batches = 0
        t0 = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        micro = 0
        for batch in train_loader:
            batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
            labels = batch.pop('labels')
            with autocast_ctx:
                logits = model(**batch).logits
                loss = loss_fn(logits, labels) / GRADIENT_ACCUMULATION_STEPS
            loss.backward()
            micro += 1
            epoch_train_loss += float(loss.item()) * GRADIENT_ACCUMULATION_STEPS
            epoch_batches += 1
            if micro % GRADIENT_ACCUMULATION_STEPS == 0:
                if MAX_GRAD_NORM is not None:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        if micro % GRADIENT_ACCUMULATION_STEPS != 0:
            if MAX_GRAD_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
        train_time_s += time.perf_counter() - t0
        train_loss_last = epoch_train_loss / max(epoch_batches, 1)

        model.eval()
        probs_all = []
        val_loss_sum = 0.0
        val_batches = 0
        t1 = time.perf_counter()
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
                labels = batch.pop('labels')
                with autocast_ctx:
                    logits = model(**batch).logits
                    probs = torch.sigmoid(logits)
                    val_loss_sum += float(loss_fn(logits, labels).item())
                probs_all.append(probs.float().cpu().numpy())
                val_batches += 1
        inference_time_s += time.perf_counter() - t1
        val_loss_last = val_loss_sum / max(val_batches, 1)
        epochs_ran = epoch

        print(f'  epoch {epoch}/{MAX_EPOCHS}  train_loss={train_loss_last:.4f}  val_loss={val_loss_last:.4f}  lr={scheduler.get_last_lr()[0]:.2e}')

        improved = val_loss_last < (best_val_loss - EARLY_STOP_MIN_DELTA)
        if improved:
            best_val_loss = val_loss_last
            best_epoch = epoch
            best_state_cpu = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = EARLY_STOP_PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                early_stopped = True
                print(f'  early stopping at epoch {epoch} (best_epoch={best_epoch}, best_val_loss={best_val_loss:.4f})')
                break

    if best_state_cpu is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state_cpu.items()})

    model.eval()
    probs_all = []
    val_loss_sum = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE, non_blocking=pin) for k, v in batch.items()}
            labels = batch.pop('labels')
            with autocast_ctx:
                logits = model(**batch).logits
                probs = torch.sigmoid(logits)
                val_loss_sum += float(loss_fn(logits, labels).item())
            probs_all.append(probs.float().cpu().numpy())
            val_batches += 1
    val_loss = val_loss_sum / max(val_batches, 1)

    prob_val = np.concatenate(probs_all, axis=0)
    pred_05 = (prob_val >= 0.5).astype(np.int64)
    per_label_05_df, summary_05 = multilabel_evaluation_report(y_val, pred_05, prob_val, LABEL_COLUMNS)

    best_thresholds, thresholds_df = tune_per_label_thresholds(y_val, prob_val, LABEL_COLUMNS, THRESHOLD_GRID)
    pred_tuned = apply_thresholds(prob_val, LABEL_COLUMNS, best_thresholds)
    per_label_tuned_df, summary_tuned = multilabel_evaluation_report(y_val, pred_tuned, prob_val, LABEL_COLUMNS)

    per_label_df = per_label_05_df.rename(
        columns={
            'precision': 'precision_baseline',
            'recall': 'recall_baseline',
            'f1': 'f1_baseline',
            'roc_auc': 'roc_auc_baseline',
        }
    ).merge(
        per_label_tuned_df.rename(
            columns={
                'precision': 'precision_tuned',
                'recall': 'recall_tuned',
                'f1': 'f1_tuned',
                'roc_auc': 'roc_auc_tuned',
            }
        ),
        on='label',
        how='left',
    )
    per_label_df['train_size'] = train_size
    for label in LABEL_COLUMNS:
        per_label_df[f'pos_weight_{label}'] = pos_weight_dict[label]

    thresholds_df['train_size'] = train_size

    row = {
        'train_size': train_size,
        'max_epochs': MAX_EPOCHS,
        'epochs_ran': epochs_ran,
        'best_epoch': best_epoch,
        'early_stopped': early_stopped,
        'warmup_ratio': WARMUP_RATIO,
        'warmup_steps': warmup_steps,
        'early_stop_patience': EARLY_STOP_PATIENCE,
        'early_stop_min_delta': EARLY_STOP_MIN_DELTA,
        'weight_decay': WEIGHT_DECAY,
        'grad_clip': MAX_GRAD_NORM,
        'grad_accum': GRADIENT_ACCUMULATION_STEPS,
        'use_bf16_amp': USE_AMP,
        'torch_compile': bool(USE_TORCH_COMPILE),
        'num_workers': NUM_WORKERS,
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'num_parameters': num_params,
        'best_val_loss': best_val_loss,
        'train_loss': train_loss_last,
        'val_loss': val_loss,
        'train_time_s': train_time_s,
        'inference_time_s': inference_time_s,
        'f1_micro_baseline': summary_05['f1_micro'],
        'f1_macro_baseline': summary_05['f1_macro'],
        'f1_micro_tuned': summary_tuned['f1_micro'],
        'f1_macro_tuned': summary_tuned['f1_macro'],
        'pos_weight_min': float(pos_weight.min().item()),
        'pos_weight_max': float(pos_weight.max().item()),
        'pos_weight_mean': float(pos_weight.mean().item()),
    }

    pd.DataFrame([row]).to_csv(row_file, index=False)
    per_label_df.to_csv(per_label_file, index=False)
    thresholds_df.to_csv(thresholds_file, index=False)

    summary_rows.append(row)
    per_label_frames.append(per_label_df)
    threshold_frames.append(thresholds_df)

benchmark_summary_df = pd.DataFrame(summary_rows).sort_values('train_size').reset_index(drop=True)
benchmark_per_label_df = pd.concat(per_label_frames, ignore_index=True)
benchmark_thresholds_df = pd.concat(threshold_frames, ignore_index=True)

benchmark_summary_df['f1_micro_delta_tuned_minus_baseline'] = (
    benchmark_summary_df['f1_micro_tuned'] - benchmark_summary_df['f1_micro_baseline']
)
benchmark_summary_df['f1_macro_delta_tuned_minus_baseline'] = (
    benchmark_summary_df['f1_macro_tuned'] - benchmark_summary_df['f1_macro_baseline']
)

summary_csv = ARTIFACT_DIR / 'distilbert_exp_b_size_benchmark_summary.csv'
per_label_csv = ARTIFACT_DIR / 'distilbert_exp_b_size_benchmark_per_label.csv'
thresholds_csv = ARTIFACT_DIR / 'distilbert_exp_b_size_benchmark_thresholds.csv'

benchmark_summary_df.to_csv(summary_csv, index=False)
benchmark_per_label_df.to_csv(per_label_csv, index=False)
benchmark_thresholds_df.to_csv(thresholds_csv, index=False)

print('Saved:')
print(' ', summary_csv)
print(' ', per_label_csv)
print(' ', thresholds_csv)
display(benchmark_summary_df)


In [ ]:
# Plots: metric changes vs training size
plt.figure(figsize=(8, 4.5))
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['train_loss'], marker='o', label='train_loss (last epoch ran)')
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['val_loss'], marker='o', label='val_loss (after best ckpt)')
plt.title('Loss vs Train Size (Experiment B)')
plt.xlabel('Train size')
plt.ylabel('Loss')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['f1_micro_baseline'], marker='o', label='micro F1 (0.5)')
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['f1_macro_baseline'], marker='o', label='macro F1 (0.5)')
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['f1_micro_tuned'], marker='o', linestyle='--', label='micro F1 (tuned)')
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['f1_macro_tuned'], marker='o', linestyle='--', label='macro F1 (tuned)')
plt.title('Aggregate F1 vs Train Size')
plt.xlabel('Train size')
plt.ylabel('F1')
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4.5))
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['train_time_s'], marker='o', label='train_time_s')
plt.plot(benchmark_summary_df['train_size'], benchmark_summary_df['inference_time_s'], marker='o', label='inference_time_s')
plt.title('Runtime vs Train Size')
plt.xlabel('Train size')
plt.ylabel('Seconds')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5.5))
for label in LABEL_COLUMNS:
    df_l = benchmark_per_label_df.loc[benchmark_per_label_df['label'] == label].sort_values('train_size')
    plt.plot(df_l['train_size'], df_l['f1_tuned'], marker='o', label=f'{label} (tuned)')
plt.title('Per-label Tuned F1 vs Train Size')
plt.xlabel('Train size')
plt.ylabel('F1')
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend(loc='best')
plt.show()
